# 1. Tensores y datos

**Objetivo.** Representar datos numéricos con tensores y dominar las operaciones necesarias para construir modelos.

**Al terminar podrás:** crear tensores; razonar sobre forma, eje y tipo; seleccionar y transformar datos; reconocer *broadcasting*; y preparar un conjunto tabular sencillo.

Un tensor es una colección rectangular de números. Un escalar tiene orden 0, un vector orden 1, una matriz orden 2 y, a partir de ahí, hablamos de tensores de orden superior. La propiedad `shape` describe cuántos elementos hay en cada eje.

In [1]:
import random
import torch
import matplotlib.pyplot as plt

SEMILLA = 42
random.seed(SEMILLA)
torch.manual_seed(SEMILLA)
torch.set_printoptions(precision=3, sci_mode=False)
print(f"PyTorch {torch.__version__} · semilla {SEMILLA}")

PyTorch 2.14.0 · semilla 42


In [2]:
escalar = torch.tensor(3.0)
vector = torch.tensor([1.0, 2.0, 3.0])
matriz = torch.arange(12, dtype=torch.float32).reshape(3, 4)
imagenes = torch.zeros(8, 3, 32, 32)

for nombre, x in {"escalar": escalar, "vector": vector, "matriz": matriz, "lote": imagenes}.items():
    print(f"{nombre:7s}: shape={tuple(x.shape)}, ndim={x.ndim}, dtype={x.dtype}")
matriz

escalar: shape=(), ndim=0, dtype=torch.float32
vector : shape=(3,), ndim=1, dtype=torch.float32
matriz : shape=(3, 4), ndim=2, dtype=torch.float32
lote   : shape=(8, 3, 32, 32), ndim=4, dtype=torch.float32


tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11.]])

## Indexación, reducción y cambio de forma

Los índices eligen observaciones o características. Las reducciones (`sum`, `mean`, `max`) eliminan un eje, salvo que pidamos conservarlo con `keepdim=True`. Cambiar la forma no cambia los datos: solo su organización lógica.

In [3]:
print("primera fila:", matriz[0])
print("última columna:", matriz[:, -1])
print("media global:", matriz.mean())
print("media por columna:", matriz.mean(dim=0))
print("transpuesta:\n", matriz.T)

primera fila: tensor([0., 1., 2., 3.])
última columna: tensor([ 3.,  7., 11.])
media global: tensor(5.500)
media por columna: tensor([4., 5., 6., 7.])
transpuesta:
 tensor([[ 0.,  4.,  8.],
        [ 1.,  5.,  9.],
        [ 2.,  6., 10.],
        [ 3.,  7., 11.]])


## *Broadcasting*

PyTorch puede combinar tensores de formas distintas cuando sus dimensiones, comparadas desde la derecha, son iguales o una de ellas vale 1. Aquí sumamos un sesgo distinto a cada columna sin copiarlo por todas las filas.

In [4]:
sesgo = torch.tensor([10.0, 20.0, 30.0, 40.0])
resultado = matriz + sesgo
print("formas:", matriz.shape, "+", sesgo.shape, "->", resultado.shape)
resultado

formas: torch.Size([3, 4]) + torch.Size([4]) -> torch.Size([3, 4])


tensor([[10., 21., 32., 43.],
        [14., 25., 36., 47.],
        [18., 29., 40., 51.]])

## De una tabla a características y objetivo

Cada fila representa una observación; cada columna, una característica. Separamos una matriz $X$ de entradas y un vector $y$ de objetivos. Estandarizar evita que una escala numérica domine a las demás.

In [5]:
datos = torch.tensor([
    [45.0, 1.0, 210.0],
    [62.0, 2.0, 290.0],
    [80.0, 3.0, 360.0],
    [95.0, 3.0, 410.0],
])
X, y = datos[:, :2], datos[:, 2]
media, escala = X.mean(dim=0), X.std(dim=0)
X_estandarizado = (X - media) / escala
print("X:\n", X)
print("y:", y)
print("medias tras estandarizar:", X_estandarizado.mean(dim=0))

X:
 tensor([[45.,  1.],
        [62.,  2.],
        [80.,  3.],
        [95.,  3.]])
y: tensor([210., 290., 360., 410.])
medias tras estandarizar: tensor([-0.000,  0.000])


## Práctica

1. Crea un tensor de forma `(2, 3, 4)` con los enteros del 0 al 23.
2. Calcula su media sobre el último eje y anticipa la forma del resultado.
3. Comprueba que la desviación típica de cada columna estandarizada es 1.
4. Explica por qué no deberíamos estandarizar el objetivo usando información del conjunto de prueba.

**Idea clave:** antes de depurar una red neuronal, comprueba siempre formas, tipos y escalas.